In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile
import os

ZIP_PATH = "/content/drive/My Drive/AIO_Homework/RCNN/data/GARBAGE CLASSIFICATION.zip"
EXTRACT_TO = "/content/data/"

os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_TO)

print("Giải nén xong!")
print(os.listdir(EXTRACT_TO))



Giải nén xong!
['GARBAGE CLASSIFICATION']


In [5]:
DATA_ROOT = "/content/data/GARBAGE CLASSIFICATION"
# Kiểm tra
print(os.listdir(DATA_ROOT))

['train', 'valid', 'test', 'data.yaml']


In [7]:
pip install selectivesearch

  Preparing metadata (setup.py) ... done
  Created wheel for selectivesearch: filename=selectivesearch-0.4-py3-none-any.whl size=4336 sha256=524fa60c7522a7fda8f51f9e73e10082f72c435cb2c72ba17db14108200a3d3e
  Stored in directory: /root/.cache/pip/wheels/7f/9b/c7/58b71f1e9fe4aa0ef8affd1c673f8818bc22a5091ea8cbbe93
Successfully built selectivesearch


In [8]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import cv2
import selectivesearch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [9]:
class RCNN(nn.Module):
    def __init__(self,num_classes):
        super(RCNN,self).__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor=nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Linear(512, num_classes + 1)  # +1 cho background
    def extract_features(self, x):
        with torch.no_grad():
            features = self.feature_extractor(x)   # (batch, 512, 1, 1)
            features = features.squeeze(-1).squeeze(-1)  # (batch, 512)
        return features
    def classify(self, features):
        return self.classifier(features)  # (batch, num_classes+1)
    def forward(self, regions):
        features = self.feature_extractor(regions)        # (N, 512, 1, 1)
        features = features.squeeze(-1).squeeze(-1)       # (N, 512)
        logits = self.classifier(features)                # (N, num_classes+1)
        return logits

In [10]:
def get_region_proposals(image_np, max_proposals=2000):
    _, regions = selectivesearch.selective_search(
        image_np,
        scale=500,
        sigma=0.9,
        min_size=10
    )

    proposals = []
    seen = set()
    for region in regions:
        x, y, w, h = region['rect']
        if w < 20 or h < 20:
            continue
        # Loại trùng
        if (x, y, w, h) in seen:
            continue
        seen.add((x, y, w, h))
        proposals.append((x, y, w, h))
        if len(proposals) >= max_proposals:
            break

    return proposals


In [11]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def crop_and_preprocess(image_pil, proposals, batch_size=32):

    crops = []
    for (x, y, w, h) in proposals:
        # Crop region từ ảnh gốc
        region = image_pil.crop((x, y, x + w, y + h))
        tensor = transform(region)   # (3, 224, 224)
        crops.append(tensor)

    # Chia thành batches
    batches = []
    for i in range(0, len(crops), batch_size):
        batch = torch.stack(crops[i:i+batch_size])  # (B, 3, 224, 224)
        batches.append(batch)

    return batches


In [12]:
def rcnn_inference(model, image_path, class_names, device, threshold=0.7):

    model.eval()
    model.to(device)

    image_pil = Image.open(image_path).convert("RGB")
    image_np  = np.array(image_pil)

    print("Đang chạy Selective Search...")
    proposals = get_region_proposals(image_np, max_proposals=50)
    print(f"  → {len(proposals)} proposals được tạo ra")

    batches = crop_and_preprocess(image_pil, proposals, batch_size=64)

    all_logits = []
    with torch.no_grad():
        for batch in batches:
            batch = batch.to(device)
            logits = model(batch)         # (B, num_classes+1)
            all_logits.append(logits.cpu())

    all_logits  = torch.cat(all_logits, dim=0)   # (N, num_classes+1)
    probs       = torch.softmax(all_logits, dim=1)
    scores, predicted_classes = probs.max(dim=1) # (N,)

    results = []
    for i, (x, y, w, h) in enumerate(proposals):
        pred_cls = predicted_classes[i].item()
        score    = scores[i].item()
        if pred_cls == 0:          # background → bỏ
            continue
        if score < threshold:      # confidence thấp → bỏ
            continue
        results.append({
            "box": (x, y, w, h),
            "class": class_names[pred_cls - 1],
            "score": score
        })

    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image_np)
    for r in results:
        x, y, w, h = r["box"]
        rect = patches.Rectangle((x, y), w, h,
                                  linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 5, f"{r['class']} {r['score']:.2f}",
                color='white', fontsize=8,
                bbox=dict(facecolor='red', alpha=0.5))
    ax.set_title(f"R-CNN: {len(results)} detections")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"\nPhát hiện {len(results)} object(s):")
    for r in results:
        print(f"  Class: {r['class']:15s} | Score: {r['score']:.4f} | Box: {r['box']}")

    return results


In [19]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Dùng device: {device}")

    # ---- Khởi tạo mô hình ----
    # Sửa CLASS_NAMES để khớp với dataset rác
    CLASS_NAMES = ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
    NUM_CLASSES = len(CLASS_NAMES)

    model = RCNN(num_classes=NUM_CLASSES)
    print(model)

    # ---- Sanity check với dummy input ----
    dummy_regions = torch.randn(4, 3, 224, 224)  # 4 region proposals giả
    output = model(dummy_regions)
    print(f"\nInput (regions): {dummy_regions.shape}")
    print(f"Output (logits): {output.shape}")   # Kỳ vọng: (4, NUM_CLASSES+1)

    # ---- Đếm tham số ----
    total_params = sum(p.numel() for p in model.parameters())
    trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal params:     {total_params:,}")
    print(f"Trainable params: {trainable:,}")

Dùng device: cuda
RCNN(
  (feature_extractor): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, 

In [20]:
import os
import glob
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = "/content/data/GARBAGE CLASSIFICATION"
CLASS_NAMES = ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']

def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = int((cx - w/2) * img_w)
    y1 = int((cy - h/2) * img_h)
    x2 = int((cx + w/2) * img_w)
    y2 = int((cy + h/2) * img_h)
    return max(0,x1), max(0,y1), min(img_w,x2), min(img_h,y2)

class GarbageRCNNDataset(Dataset):

    def __init__(self, split="train", transform=None,
                 max_proposals=50, iou_pos=0.5, iou_neg=0.3):
        self.transform    = transform
        self.max_proposals = max_proposals
        self.iou_pos      = iou_pos
        self.iou_neg      = iou_neg

        img_dir   = os.path.join(DATA_ROOT, split, "images")
        label_dir = os.path.join(DATA_ROOT, split, "labels")

        self.img_paths   = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        self.img_paths  += sorted(glob.glob(os.path.join(img_dir, "*.jpeg")))
        self.label_dir   = label_dir


        print(f"[{split}] Tìm thấy {len(self.img_paths)} ảnh")

    def _load_gt_boxes(self, label_path, img_w, img_h):
        boxes, labels = [], []
        if not os.path.exists(label_path):
            return boxes, labels
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, w, h = map(float, parts[1:5])
                x1, y1, x2, y2 = yolo_to_xyxy(cx, cy, w, h, img_w, img_h)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls_id + 1)  # +1 vì 0 là background
        return boxes, labels

    def _compute_iou(self, box1, box2):
        xi1 = max(box1[0], box2[0])
        yi1 = max(box1[1], box2[1])
        xi2 = min(box1[2], box2[2])
        yi2 = min(box1[3], box2[3])
        inter = max(0, xi2-xi1) * max(0, yi2-yi1)
        area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
        area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img_pil  = Image.open(img_path).convert("RGB")
        img_w, img_h = img_pil.size

        # Load GT boxes
        stem       = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.label_dir, stem + ".txt")
        gt_boxes, gt_labels = self._load_gt_boxes(label_path, img_w, img_h)

        # Selective Search để tạo proposals
        img_np    = np.array(img_pil)
        proposals = get_region_proposals(img_np, max_proposals=self.max_proposals)

        crops, labels = [], []
        for (x, y, w, h) in proposals:
            prop_box = [x, y, x+w, y+h]

            # Tính IoU với từng GT box → gán nhãn
            best_iou, best_label = 0, 0
            for gt_box, gt_label in zip(gt_boxes, gt_labels):
                iou = self._compute_iou(prop_box, gt_box)
                if iou > best_iou:
                    best_iou, best_label = iou, gt_label

            if best_iou >= self.iou_pos:
                label = best_label      # positive: class đúng
            elif best_iou <= self.iou_neg:
                label = 0               # background
            else:
                continue

            # Crop và transform
            region = img_pil.crop((x, y, x+w, y+h))
            if self.transform:
                region = self.transform(region)
            crops.append(region)
            labels.append(label)

        if len(crops) == 0:
            # Trả về 1 crop giả nếu không có proposal nào
            dummy = torch.zeros(3, 224, 224)
            return dummy.unsqueeze(0), torch.tensor([0])

        crops  = torch.stack(crops)           # (N, 3, 224, 224)
        labels = torch.tensor(labels)         # (N,)
        return crops, labels

In [ ]:
from tqdm import tqdm

def train_rcnn(model, num_epochs=5, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training trên: {device}")

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    train_dataset = GarbageRCNNDataset(split="train",
                                       transform=train_transform,
                                       max_proposals=100)

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(num_epochs):
        total_loss, total_correct, total_samples = 0, 0, 0

        # ← tqdm bọc vào đây để có progress bar
        pbar = tqdm(range(len(train_dataset)),
                    desc=f"Epoch {epoch+1}/{num_epochs}",
                    unit="img")

        for img_idx in pbar:
            crops, labels = train_dataset[img_idx]
            if crops.shape[0] == 0:
                continue

            crops  = crops.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(crops)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += len(labels)
            total_loss    += loss.item()

            # Cập nhật thông tin trên progress bar
            pbar.set_postfix({
                "loss": f"{total_loss / (img_idx + 1):.4f}",
                "acc":  f"{total_correct / max(total_samples, 1):.3f}"
            })

        print(f"✅ Epoch {epoch+1} | "
              f"Loss: {total_loss/len(train_dataset):.4f} | "
              f"Acc: {total_correct/max(total_samples,1):.3f}\n")

    return model

# Chạy
if __name__ == "__main__":
    model = RCNN(num_classes=len(CLASS_NAMES))
    trained_model = train_rcnn(model, num_epochs=3, lr=1e-4)
    torch.save(trained_model.state_dict(), "rcnn_garbage.pth")
    print("Đã lưu model!")


Training trên: cuda
[train] Tìm thấy 7324 ảnh


Epoch 1/3:   0%|          | 0/7324 [00:00<?, ?img/s]/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
Epoch 1/3:   0%|          | 19/7324 [00:34<3:34:33,  1.76s/img, loss=1.8554, acc=0.406]/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
Epoch 1/3:   0%|          | 24/7324 [00:42<3:17:43,  1.63s/img, loss=1.7428, acc=0.488]/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to fl